<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/rag-production-hardening/module-04-rag/lesson-4.1-document-ai/practice/GCP_Capstone_4.1_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 4.1 — Document AI — OCR, Layout Parser, Form Parser

Runnable companion to the published practice lab: every exercise with a complete solution grounded in the lesson notebook. Run the **Setup** cell first, then work through the exercises. Cloud Shell / `gcloud` steps are `%%bash` cells.

---

## Setup

Install the Document AI + Firestore SDKs, authenticate with Application Default Credentials (no API keys), and set your project. The three processors (OCR, Layout Parser, Form Parser) are created below if they are missing.

**The documents are real.** Two of the thirteen real documents in DocuMind's corpus (`deploy/evals/real_sources.json`): the **POSH Act, 2013** as a *scanned* Gazette of India (page images, no text layer) for the OCR exercises, and the **Payment of Gratuity Act, 1972** as the ministry's text PDF for the Layout Parser and the ingester. They come from the course repo beside this notebook, or from the publisher with the sha256 the kit recorded. Only the onboarding form stays synthetic — a filled form is personal data by definition.


In [ ]:
!pip install -q google-cloud-documentai google-cloud-documentai-toolbox google-cloud-firestore google-genai reportlab pypdf
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE
LOCATION = 'us'  # Document AI location: 'us' or 'eu'
TENANT_ID = 'acme'  # the teaching tenant; every chunk this lab writes carries it

# --- 1. The real documents (deploy/evals/real_sources.json): kit first, publisher second ---
import hashlib, os, subprocess, urllib.request
REAL = {   # slug: (where the ministry / state government publishes it, sha256 of the bytes the kit recorded)
    'posh_act_2013': ('https://gil.gujarat.gov.in/Media/DocumentUpload/posh_act._english.pdf',
                      '70d525e419baf7a9d249d9c679a23714603509d018d38f932be3a18365d3b195'),      # 13 pages, a SCANNED Gazette: no text layer
    'payment_of_gratuity_act_1972': ('https://www.labour.gov.in/static/uploads/2025/06/072a4b7ea8246533c62b96b68a30da53.pdf',
                                     'fa044c91c6714c738ded7ba98dbe9950060d9bf85ec2a63996cee927269669d4'),      # 10 pages, text PDF
}

def real_document(slug):
    """The PDF from the kit beside this notebook (or its clone under /content); else from the publisher."""
    for base in ('deploy/evals/corpus/acme', '../deploy/evals/corpus/acme',
                 '/content/agentic-ai-weekend-gcp-learners/deploy/evals/corpus/acme'):
        if os.path.isfile(f'{base}/{slug}.pdf'):
            return f'{base}/{slug}.pdf'
    if not os.path.isdir('/content/agentic-ai-weekend-gcp-learners'):
        subprocess.run(['git', 'clone', '--depth', '1', '-b', 'main', 'https://github.com/netsetos/agentic-ai-weekend-gcp-learners',
                        '/content/agentic-ai-weekend-gcp-learners'], check=False)
    if os.path.isfile(f'/content/agentic-ai-weekend-gcp-learners/deploy/evals/corpus/acme/{slug}.pdf'):
        return f'/content/agentic-ai-weekend-gcp-learners/deploy/evals/corpus/acme/{slug}.pdf'
    url, sha = REAL[slug]
    data = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'}),
                                  timeout=180).read()
    if hashlib.sha256(data).hexdigest() != sha:
        print(f'  {slug}: the publisher changed this file since the kit recorded it - read it before trusting it')
    with open(f'{slug}.pdf', 'wb') as f:
        f.write(data)
    return f'{slug}.pdf'

POSH_PDF = real_document('posh_act_2013')                     # Exercises 1, 2, 6: OCR on a scan
GRATUITY_PDF = real_document('payment_of_gratuity_act_1972')  # Exercises 4, 7, 8: layout, chunks, ingest
print(f'Real documents: {POSH_PDF} ({os.path.getsize(POSH_PDF):,} bytes), '
      f'{GRATUITY_PDF} ({os.path.getsize(GRATUITY_PDF):,} bytes)')

# --- 1b. A synthetic form.pdf (labelled fields + a table) for the Form Parser (Exercise 5). A
#         filled form is personal data by definition, so this one is drawn, not downloaded. ---
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
_f = canvas.Canvas('form.pdf', pagesize=letter)
_f.setFont('Helvetica-Bold', 15); _f.drawString(72, 730, 'DocuMind AI - Onboarding Form')
_f.setFont('Helvetica', 11)
for _i, _line in enumerate([
        'Full Name: Priya Sharma', 'Employee ID: EMP-2026-118',
        'Department: Machine Learning', 'Date of Joining: 2026-09-01',
        'Location: Hyderabad', 'Email: priya.sharma@acme.in',
        'Manager: Rahul Verma']):
    _f.drawString(72, 700 - _i * 24, _line)
_f.setFont('Helvetica-Bold', 11); _f.drawString(72, 510, 'Equipment Issued')
_y = 490
for _r, (_a, _b) in enumerate([('Item', 'Serial'), ('Laptop', 'LT-9921'),
                               ('Monitor', 'MN-4415'), ('Access Card', 'AC-7788')]):
    _f.setFont('Helvetica-Bold' if _r == 0 else 'Helvetica', 10)
    _f.drawString(80, _y, _a); _f.drawString(260, _y, _b)
    _f.line(72, _y - 5, 400, _y - 5)
    _y -= 22
_f.showPage(); _f.save()
print('Wrote form.pdf')

# --- 2. Document AI processors: created once if missing (or paste your own IDs) ---
from google.api_core.client_options import ClientOptions
from google.cloud import documentai
_da = documentai.DocumentProcessorServiceClient(
    client_options=ClientOptions(api_endpoint=f'{LOCATION}-documentai.googleapis.com'))
_parent = _da.common_location_path(PROJECT_ID, LOCATION)

def _ensure_processor(display_name, type_):
    # reuse an existing processor of this type, else create one (needs the Document
    # AI API enabled + documentai.processors.create; else create in the Console and
    # paste the ID here).
    for p in _da.list_processors(parent=_parent):
        if p.type_ == type_:
            return p.name.split('/')[-1]
    p = _da.create_processor(parent=_parent,
        processor=documentai.Processor(type_=type_, display_name=display_name))
    return p.name.split('/')[-1]

OCR_ID    = _ensure_processor('documind-ocr',    'OCR_PROCESSOR')
LAYOUT_ID = _ensure_processor('documind-layout', 'LAYOUT_PARSER_PROCESSOR')
FORM_ID   = _ensure_processor('documind-form',   'FORM_PARSER_PROCESSOR')
print(f'Processors ready: OCR={OCR_ID}  LAYOUT={LAYOUT_ID}  FORM={FORM_ID}')

# --- 3. Firestore (default) database for Cell 5 (created once if missing) ---
import subprocess
FIRESTORE_LOCATION = 'asia-south1'  # region for the (default) DB; PERMANENT once created
if '(default)' not in subprocess.run(
        ['gcloud', 'firestore', 'databases', 'list', '--project', PROJECT_ID, '--format=value(name)'],
        capture_output=True, text=True).stdout:
    print(f'Creating Firestore (default) database in {FIRESTORE_LOCATION} (one-time)...')
    subprocess.run(['gcloud', 'firestore', 'databases', 'create',
                    '--location=' + FIRESTORE_LOCATION, '--project', PROJECT_ID], check=False)
else:
    print('Firestore (default) database ready.')

USD_INR = 85  # for any cost display

### Universal processing function

Every exercise routes through this one helper: build a `ProcessRequest` with a `RawDocument`, call `process_document()`, return the parsed `Document`. The regional `api_endpoint` must match `LOCATION`.

In [ ]:
from google.api_core.client_options import ClientOptions
from google.cloud import documentai

def process_document(project_id, location, processor_id, file_path,
                     mime_type='application/pdf', process_options=None):
    client = documentai.DocumentProcessorServiceClient(
        client_options=ClientOptions(
            api_endpoint=f'{location}-documentai.googleapis.com'))
    name = client.processor_path(project_id, location, processor_id)
    with open(file_path, 'rb') as f:
        content = f.read()
    request = documentai.ProcessRequest(
        name=name,
        raw_document=documentai.RawDocument(content=content, mime_type=mime_type),
        process_options=process_options)
    return client.process_document(request=request).document

print('process_document() ready')

## Exercise 1: First OCR Call

**Difficulty:** Easy

OCR the scanned POSH Act (13 pages fits the 15-page online cap). Print the full text and the page-1 token count — and, first, what `pypdf` gets from the same file.

1. Show that pypdf returns noise for a scan (that is why the OCR lane exists)
2. Call process_document() with the OCR processor
3. Print doc.text and count doc.pages[0].tokens


In [ ]:
# The scanned POSH Act: page images, no text layer. First, what a PDF library sees.
from pypdf import PdfReader
_pypdf = (PdfReader(POSH_PDF).pages[2].extract_text() or '')
print(f'pypdf, page 3: {len(_pypdf)} chars, e.g. {_pypdf[:80]!r}')

options = documentai.ProcessOptions(
    ocr_config=documentai.OcrConfig(
        enable_native_pdf_parsing=True))   # use a text layer when there is one; there is none here

doc = process_document(PROJECT_ID, LOCATION, OCR_ID, POSH_PDF,
                       process_options=options)

print(f'\nDocument AI OCR: {len(doc.text)} chars from {len(doc.pages)} pages')
print(doc.text[:500])

token_count = len(doc.pages[0].tokens)
print(f'\nPage 1 token count: {token_count}')


## Exercise 2: Page-Level Details

**Difficulty:** Easy

Process the scanned POSH Act again with quality scores. Print languages, paragraph count and quality per page — on a scan the quality score means something.

1. Use enable_image_quality_scores=True
2. Loop through doc.pages
3. Print detected_languages and quality_score

In [ ]:
options = documentai.ProcessOptions(
    ocr_config=documentai.OcrConfig(
        enable_native_pdf_parsing=True,
        enable_image_quality_scores=True))

doc = process_document(PROJECT_ID, LOCATION, OCR_ID, POSH_PDF,
                       process_options=options)

print(f'Pages: {len(doc.pages)}')
def _anchor_text(document, text_anchor):
    # Online process() leaves text_anchor.content empty; resolve text from document.text
    if not text_anchor.text_segments:
        return (text_anchor.content or '').strip()
    return ''.join(document.text[int(s.start_index):int(s.end_index)]
                   for s in text_anchor.text_segments).strip()

for page in doc.pages:
    langs = [(l.language_code, f'{l.confidence:.0%}') for l in page.detected_languages]
    quality = page.image_quality_scores.quality_score
    print(f'  Page {page.page_number}: {len(page.paragraphs)} paragraphs, '
          f'langs={langs}, quality={quality:.2f}')

## Exercise 3: Hindi OCR

**Difficulty:** Easy

Process a Hindi document with language_hints=["hi"]. Verify Devanagari extraction.

1. Set hints in OcrConfig
2. Process Hindi PDF/image
3. Verify Devanagari script in output

In [ ]:
# Upload a Hindi PDF/image as 'hindi.pdf' first.
options = documentai.ProcessOptions(
    ocr_config=documentai.OcrConfig(
        enable_native_pdf_parsing=True,
        hints=documentai.OcrConfig.Hints(language_hints=['hi'])))

doc = process_document(PROJECT_ID, LOCATION, OCR_ID, 'hindi.pdf',
                       process_options=options)

print('Extracted text (first 500 chars):')
print(doc.text[:500])

# Verify Devanagari (Unicode block U+0900–U+097F) is present
has_devanagari = any('ऀ' <= ch <= 'ॿ' for ch in doc.text)
print(f'\nDevanagari script present: {has_devanagari}')
for page in doc.pages:
    langs = [(l.language_code, f'{l.confidence:.0%}') for l in page.detected_languages]
    print(f'  Page {page.page_number} detected_languages: {langs}')

## Exercise 4: Layout Parser Chunking

**Difficulty:** Medium

Process the Payment of Gratuity Act, 1972 (10 pages, text PDF) with Layout Parser. Print chunks with their ancestor headings and page spans.

1. Set include_ancestor_headings=True, chunk_size=1024
2. Process GRATUITY_PDF with Layout Parser
3. Print each chunk with its heading context

In [ ]:
options = documentai.ProcessOptions(
    layout_config=documentai.ProcessOptions.LayoutConfig(
        enable_table_annotation=True,
        enable_image_annotation=True,
        chunking_config=documentai.ProcessOptions.LayoutConfig.ChunkingConfig(
            chunk_size=1024,
            include_ancestor_headings=True)))

doc = process_document(PROJECT_ID, LOCATION, LAYOUT_ID, GRATUITY_PDF,
                       process_options=options)

print(f'Chunks: {len(doc.chunked_document.chunks)} from {GRATUITY_PDF}')
for i, chunk in enumerate(doc.chunked_document.chunks[:5]):
    print(f'\n--- Chunk {i} ({chunk.chunk_id}) ---')
    print(f'Pages: {chunk.page_span.page_start}-{chunk.page_span.page_end}')
    print(f'Content:\n{chunk.content[:200]}...')

## Exercise 5: Form Parser KVP

**Difficulty:** Medium

Process an invoice or form PDF. Extract key-value pairs and tables.

1. Process with Form Parser
2. Loop page.form_fields for KVPs
3. Loop page.tables for structured data

In [ ]:
# Upload an invoice/form PDF as 'form.pdf' first.
doc = process_document(PROJECT_ID, LOCATION, FORM_ID, 'form.pdf')

for page in doc.pages:
    print(f'\n=== Page {page.page_number} ===')
    for field in page.form_fields:
        key = _anchor_text(doc, field.field_name.text_anchor)
        val = _anchor_text(doc, field.field_value.text_anchor)
        print(f'  {key}: {val} ({field.field_value.confidence:.0%})')
    for idx, table in enumerate(page.tables):
        print(f'\n  Table {idx}:')
        for row in table.body_rows:
            cells = [_anchor_text(doc, c.layout.text_anchor) for c in row.cells]
            print(f'    {cells}')

## Exercise 6: Quality Gate Pipeline

**Difficulty:** Medium

OCR the scanned POSH Act with quality scores. Filter pages below 0.5. Only chunk high-quality pages.

1. Enable quality scores in OCR
2. Filter pages where quality_score >= 0.5
3. Only include high-quality page text in chunks

In [ ]:
# Step 1 — OCR with quality scores to find the good pages.
ocr_options = documentai.ProcessOptions(
    ocr_config=documentai.OcrConfig(
        enable_native_pdf_parsing=True,
        enable_image_quality_scores=True))

ocr_doc = process_document(PROJECT_ID, LOCATION, OCR_ID, POSH_PDF,
                           process_options=ocr_options)

QUALITY_THRESHOLD = 0.5
good_pages, bad_pages = [], []
for page in ocr_doc.pages:
    score = page.image_quality_scores.quality_score
    (good_pages if score >= QUALITY_THRESHOLD else bad_pages).append(
        (page.page_number, score))

print(f'High-quality pages (>= {QUALITY_THRESHOLD}): {good_pages}')
print(f'Rejected pages: {bad_pages}')

# Step 2 — only chunk if the document cleared the gate. Chunking is
# document-level, so we gate the whole doc on having usable pages and
# keep only chunks that fall entirely within high-quality page ranges.
good_page_numbers = {pn for pn, _ in good_pages}
if good_page_numbers:
    layout_options = documentai.ProcessOptions(
        layout_config=documentai.ProcessOptions.LayoutConfig(
            chunking_config=documentai.ProcessOptions.LayoutConfig.ChunkingConfig(
                chunk_size=1024, include_ancestor_headings=True)))
    layout_doc = process_document(PROJECT_ID, LOCATION, LAYOUT_ID, POSH_PDF,
                                  process_options=layout_options)
    kept = [c for c in layout_doc.chunked_document.chunks
            if all(p in good_page_numbers
                   for p in range(c.page_span.page_start, c.page_span.page_end + 1))]
    print(f'\nKept {len(kept)} / {len(layout_doc.chunked_document.chunks)} chunks '
          f'after quality gate')
else:
    print('\nNo pages cleared the quality gate — nothing chunked.')

## Exercise 7: Chunks to Firestore

**Difficulty:** Challenge

Layout Parser chunks of the Gratuity Act into `chunks` — DocuMind's ONE collection — as the canonical document. Verify in Console.

1. Process GRATUITY_PDF with Layout Parser
2. Store each chunk with tenant_id, text, source_uri, page_start, doc_type, embedding (768-d, text-embedding-005)
3. Verify in Firestore Console: ids like `acme:payment_of_gratuity_act_1972#d3`


In [ ]:
import os
from google import genai
from google.genai import types
from google.cloud import firestore
from google.cloud.firestore_v1.vector import Vector

CHUNKS_COLLECTION = 'chunks'   # one name, from 2.3 to production (deploy/services/rag-api/config.py)
EMBED_CONFIG = types.EmbedContentConfig(task_type='RETRIEVAL_DOCUMENT', output_dimensionality=768)
emb = genai.Client(enterprise=True, project=PROJECT_ID, location='us-central1')  # embeddings are regional-only

def embed_texts(texts):
    """Embed chunk contents in batches of at most 250 (text-embedding-005 request limit)."""
    vectors = []
    for i in range(0, len(texts), 250):
        resp = emb.models.embed_content(model='text-embedding-005', contents=texts[i:i + 250],
                                        config=EMBED_CONFIG)
        vectors.extend(e.values for e in resp.embeddings)
    return vectors

def store_chunks(document, file_path, tenant_id=TENANT_ID, doc_type='statute'):
    """The canonical document (the fields services/ingest/indexer.py writes) into chunks."""
    db = firestore.Client(project=PROJECT_ID)
    slug = os.path.basename(file_path).rsplit('.', 1)[0]
    source_uri = f'gs://{PROJECT_ID}-uploads/{tenant_id}/{os.path.basename(file_path)}'  # where upload.sh puts it
    chunks = list(document.chunked_document.chunks)
    vectors = embed_texts([c.content for c in chunks])
    batch, pending = db.batch(), 0
    for i, (chunk, vec) in enumerate(zip(chunks, vectors)):
        ref = db.collection(CHUNKS_COLLECTION).document(f'{tenant_id}:{slug}#d{i}')
        batch.set(ref, {
            'tenant_id': tenant_id, 'text': chunk.content, 'embedding': Vector(vec),
            'source_uri': source_uri,
            'page_start': chunk.page_span.page_start, 'page_end': chunk.page_span.page_end,
            'doc_type': doc_type, 'kind': 'text', 'section': None,
            'processed_at': firestore.SERVER_TIMESTAMP})
        pending += 1
        if pending == 400:   # Firestore batches hold at most 500 ops: commit early
            batch.commit()
            batch, pending = db.batch(), 0
    if pending:
        batch.commit()
    print(f'Stored {len(chunks)} embedded chunks from {file_path} as tenant {tenant_id!r}')

# Reuse the Layout-Parser output from Exercise 4 (variable `doc`), or re-run:
options = documentai.ProcessOptions(
    layout_config=documentai.ProcessOptions.LayoutConfig(
        chunking_config=documentai.ProcessOptions.LayoutConfig.ChunkingConfig(
            chunk_size=1024, include_ancestor_headings=True)))
doc = process_document(PROJECT_ID, LOCATION, LAYOUT_ID, GRATUITY_PDF,
                       process_options=options)

store_chunks(doc, GRATUITY_PDF)
# Now open Firestore Console -> 'chunks' -> the acme:payment_of_gratuity_act_1972#d* documents.


## Exercise 8: DocumentIngester Module

**Difficulty:** Challenge

Build the complete DocumentIngester class with process() and ingest_for_rag(), and make it production-shaped in two ways.

1. Implement process() for any processor
2. Implement ingest_for_rag() with Layout + embeddings + Firestore — slicing a PDF longer than the 15-page online cap with pypdf, and skipping a document the tenant already holds (tenant_id + source_uri, the same rule 4.2's loader applies)
3. Test end-to-end with the Gratuity Act: the second run must report the skip


In [ ]:
# The lab's ingester - the worker's Document AI path is deploy/services/ingest/parser.py, called from main.py on every upload.
import io
from google.cloud.firestore_v1.base_query import FieldFilter
from pypdf import PdfReader, PdfWriter

ONLINE_PAGE_LIMIT = 15   # Document AI online (synchronous) requests; batch takes 500 (OCR, Layout)

class DocumentIngester:
    def __init__(self, project_id, location='us', tenant_id=TENANT_ID):
        self.project_id = project_id
        self.location = location
        self.tenant_id = tenant_id
        self.client = documentai.DocumentProcessorServiceClient(
            client_options=ClientOptions(
                api_endpoint=f'{location}-documentai.googleapis.com'))
        self.db = firestore.Client(project=self.project_id)

    def process(self, processor_id, content, mime_type='application/pdf', process_options=None):
        name = self.client.processor_path(self.project_id, self.location, processor_id)
        request = documentai.ProcessRequest(
            name=name,
            raw_document=documentai.RawDocument(content=content, mime_type=mime_type),
            process_options=process_options)
        return self.client.process_document(request=request).document

    @staticmethod
    def slices(file_path, limit=ONLINE_PAGE_LIMIT):
        """(pdf bytes, first page number) per slice of at most `limit` pages."""
        reader = PdfReader(file_path)
        for first in range(0, len(reader.pages), limit):
            w = PdfWriter()
            for p in reader.pages[first:first + limit]:
                w.add_page(p)
            buf = io.BytesIO(); w.write(buf)
            yield buf.getvalue(), first

    def already_ingested(self, source_uri):
        return bool(self.db.collection(CHUNKS_COLLECTION)
                    .where(filter=FieldFilter('tenant_id', '==', self.tenant_id))
                    .where(filter=FieldFilter('source_uri', '==', source_uri)).limit(1).get())

    def ingest_for_rag(self, layout_id, file_path, doc_type='statute', chunk_size=1024):
        slug = os.path.basename(file_path).rsplit('.', 1)[0]
        source_uri = f'gs://{self.project_id}-uploads/{self.tenant_id}/{os.path.basename(file_path)}'
        if self.already_ingested(source_uri):
            print(f'{slug}: already in chunks for tenant {self.tenant_id!r} - skipped')
            return 0
        options = documentai.ProcessOptions(
            layout_config=documentai.ProcessOptions.LayoutConfig(
                chunking_config=documentai.ProcessOptions.LayoutConfig.ChunkingConfig(
                    chunk_size=chunk_size,
                    include_ancestor_headings=True)))
        rows = []
        for content, page_offset in self.slices(file_path):
            doc = self.process(layout_id, content, process_options=options)
            for chunk in doc.chunked_document.chunks:
                rows.append({'text': chunk.content,
                             'page_start': chunk.page_span.page_start + page_offset,
                             'page_end': chunk.page_span.page_end + page_offset})
        vectors = embed_texts([r['text'] for r in rows])
        batch, pending = self.db.batch(), 0
        for i, (r, vec) in enumerate(zip(rows, vectors)):
            ref = self.db.collection(CHUNKS_COLLECTION).document(f'{self.tenant_id}:{slug}#d{i}')
            batch.set(ref, {
                'tenant_id': self.tenant_id, 'text': r['text'], 'embedding': Vector(vec),
                'source_uri': source_uri, 'page_start': r['page_start'], 'page_end': r['page_end'],
                'doc_type': doc_type, 'kind': 'text', 'section': None,
                'processed_at': firestore.SERVER_TIMESTAMP})
            pending += 1
            if pending == 400:
                batch.commit()
                batch, pending = self.db.batch(), 0
        if pending:
            batch.commit()
        return len(rows)

print('DocumentIngester ready')


In [ ]:
# End-to-end test with the Gratuity Act. Exercise 7 already wrote it, so the FIRST line printed
# should be the skip; delete those documents in the Console and re-run to see a full ingest.
ingester = DocumentIngester(PROJECT_ID, LOCATION)
n = ingester.ingest_for_rag(LAYOUT_ID, GRATUITY_PDF)
print(f'Ingested {n} chunks into chunks as tenant {TENANT_ID!r} (0 means the skip rule fired)')
